# Week 6 — Pandas Time Series

Time series analysis of the Olist e-commerce dataset, focusing on order activity, monthly revenue and changes over time.

## 1. Data Loading

In [1]:
import pandas as pd

In [2]:
# Load orders and order items datasets

# Load Olist datasets
orders = pd.read_csv("../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")

In [3]:
print("Orders:", orders.shape)
print("Order items:", order_items.shape)

display(orders.head())
display(order_items.head())

Orders: (99441, 8)
Order items: (112650, 7)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


## 2. Datetime Preparation

In [4]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

In [5]:
orders["year"] = orders["order_purchase_timestamp"].dt.year
orders["month"] = orders["order_purchase_timestamp"].dt.month
orders["day"] = orders["order_purchase_timestamp"].dt.day

In [6]:
display(
    orders[["order_purchase_timestamp", "year", "month", "day"]].head()
)

print(orders["order_purchase_timestamp"].dtype)

,order_purchase_timestamp,year,month,day
0,2017-10-02 10:56:33,2017,10,2
1,2018-07-24 20:41:37,2018,7,24
2,2018-08-08 08:38:49,2018,8,8
3,2017-11-18 19:28:06,2017,11,18
4,2018-02-13 21:18:39,2018,2,13


datetime64[us]


# Filter orders purchased in Q2 2018

In [7]:
orders_q2_2018 = orders[
    (orders["order_purchase_timestamp"] >= '2018-04-01') &
    (orders["order_purchase_timestamp"] < '2018-07-01')
]
print("Orders in Q2 2018:", len(orders_q2_2018))

Orders in Q2 2018: 19979


# Aggregate order counts by day

In [8]:
orders_time = orders.set_index("order_purchase_timestamp")
daily_orders = (
    orders_time
    .resample("D")["order_id"]
    .count()
)
daily_orders.head(10)

order_purchase_timestamp
2016-09-04    1
2016-09-05    1
2016-09-06    0
2016-09-07    0
2016-09-08    0
2016-09-09    0
2016-09-10    0
2016-09-11    0
2016-09-12    0
2016-09-13    1
Freq: D, Name: order_id, dtype: int64

# Calculate monthly order counts for 2018

In [9]:
monthly_orders = (
    orders_time 
    .resample("MS")["order_id"]
    .count()
)

monthly_orders_2018 = monthly_orders.loc["2018-01":"2018-12"]

print(monthly_orders_2018)

order_purchase_timestamp
2018-01-01    7269
2018-02-01    6728
2018-03-01    7211
2018-04-01    6939
2018-05-01    6873
2018-06-01    6167
2018-07-01    6292
2018-08-01    6512
2018-09-01      16
2018-10-01       4
Freq: MS, Name: order_id, dtype: int64


# Compare monthly order counts with the previous month

In [10]:
monthly_analysis = (
    monthly_orders_2018
    .to_frame(name="orders")
    .loc["2018-01":"2018-08"]
)

monthly_analysis["previous_orders"] = monthly_analysis["orders"].shift(1)

display(monthly_analysis)

,orders,previous_orders
order_purchase_timestamp,,
2018-01-01,7269,NaN
2018-02-01,6728,7269.0
2018-03-01,7211,6728.0
2018-04-01,6939,7211.0
2018-05-01,6873,6939.0
2018-06-01,6167,6873.0
2018-07-01,6292,6167.0
2018-08-01,6512,6292.0


# Calculate month-over-month change in order counts

In [11]:
monthly_analysis["orders_change"] = monthly_analysis["orders"].diff()

display(monthly_analysis)

,orders,previous_orders,orders_change
order_purchase_timestamp,,,
2018-01-01,7269,NaN,NaN
2018-02-01,6728,7269.0,-541.0
2018-03-01,7211,6728.0,483.0
2018-04-01,6939,7211.0,-272.0
2018-05-01,6873,6939.0,-66.0
2018-06-01,6167,6873.0,-706.0
2018-07-01,6292,6167.0,125.0
2018-08-01,6512,6292.0,220.0


# Calculate cumulative order counts

In [12]:
monthly_analysis["cumulative_orders"] = monthly_analysis["orders"].cumsum()

display(monthly_analysis)

,orders,previous_orders,orders_change,cumulative_orders
order_purchase_timestamp,,,,
2018-01-01,7269,NaN,NaN,7269
2018-02-01,6728,7269.0,-541.0,13997
2018-03-01,7211,6728.0,483.0,21208
2018-04-01,6939,7211.0,-272.0,28147
2018-05-01,6873,6939.0,-66.0,35020
2018-06-01,6167,6873.0,-706.0,41187
2018-07-01,6292,6167.0,125.0,47479
2018-08-01,6512,6292.0,220.0,53991


In [13]:
orders_revenue = orders.merge(
    order_items,
    how='left',
    on='order_id'
)
monthly_revenue = (
    orders_revenue
    .resample("MS", on="order_purchase_timestamp")["price"]
    .sum()
    .loc['2018-01': '2018-08']
)
monthly_analysis["revenue"] = monthly_revenue

display(monthly_revenue)

order_purchase_timestamp
2018-01-01    950030.36
2018-02-01    844178.71
2018-03-01    983213.44
2018-04-01    996647.75
2018-05-01    996517.68
2018-06-01    865124.31
2018-07-01    895507.22
2018-08-01    854686.33
Freq: MS, Name: price, dtype: float64

In [14]:
# Calculate monthly revenue growth and cumulative revenue

In [15]:
monthly_analysis["previous_revenue"] = monthly_analysis["revenue"].shift(1)
monthly_analysis["revenue_change"] = monthly_analysis["revenue"].diff()
monthly_analysis["cumulative_revenue"] = monthly_analysis["revenue"].cumsum()
monthly_analysis["revenue_mom_pct"] = ((monthly_analysis["revenue"] - monthly_analysis["previous_revenue"])\
                                       / monthly_analysis["previous_revenue"]) * 100
monthly_analysis["revenue_mom_pct"] = monthly_analysis["revenue_mom_pct"].round(2)
display(monthly_analysis)

,orders,previous_orders,orders_change,cumulative_orders,revenue,previous_revenue,revenue_change,cumulative_revenue,revenue_mom_pct
order_purchase_timestamp,,,,,,,,,
2018-01-01,7269,NaN,NaN,7269,950030.36,NaN,NaN,950030.36,NaN
2018-02-01,6728,7269.0,-541.0,13997,844178.71,950030.36,-105851.65,1794209.07,-11.14
2018-03-01,7211,6728.0,483.0,21208,983213.44,844178.71,139034.73,2777422.51,16.47
2018-04-01,6939,7211.0,-272.0,28147,996647.75,983213.44,13434.31,3774070.26,1.37
2018-05-01,6873,6939.0,-66.0,35020,996517.68,996647.75,-130.07,4770587.94,-0.01
2018-06-01,6167,6873.0,-706.0,41187,865124.31,996517.68,-131393.37,5635712.25,-13.19
2018-07-01,6292,6167.0,125.0,47479,895507.22,865124.31,30382.91,6531219.47,3.51
2018-08-01,6512,6292.0,220.0,53991,854686.33,895507.22,-40820.89,7385905.80,-4.56


## Week 6 Pandas Gate

Weekly order counts and changes for Q2 2018

In [29]:

mask = (orders["order_purchase_timestamp"] >= '2018-04-01') & (orders["order_purchase_timestamp"] < '2018-07-01')
filtered_orders = orders.loc[mask]


weekly_orders = (
    filtered_orders.set_index("order_purchase_timestamp")
    .resample("W-MON", closed="left", label="left")["order_id"]
    .count()
    .reset_index()
)

weekly_orders.columns = ["week_start", "orders"]

weekly_orders["orders_change"] = weekly_orders["orders"].diff()


print(weekly_orders)

   week_start  orders  orders_change
0  2018-03-26     207            NaN
1  2018-04-02    1590         1383.0
2  2018-04-09    1554          -36.0
3  2018-04-16    1672          118.0
4  2018-04-23    1676            4.0
5  2018-04-30    1770           94.0
6  2018-05-07    1987          217.0
7  2018-05-14    1837         -150.0
8  2018-05-21     954         -883.0
9  2018-05-28    1083          129.0
10 2018-06-04    1438          355.0
11 2018-06-11    1514           76.0
12 2018-06-18    1457          -57.0
13 2018-06-25    1240         -217.0


Monthly order growth from April to August 2018

In [30]:

mask = (orders["order_purchase_timestamp"] >= '2018-04-01') & (orders["order_purchase_timestamp"] < '2018-09-01')
filtered_orders = orders.loc[mask]

monthly_growth = (
    filtered_orders.set_index("order_purchase_timestamp")
    .resample("MS")["order_id"]
    .count()
    .reset_index()
)

monthly_growth.columns = ["month", "orders"]

monthly_growth["previous_orders"] = monthly_growth["orders"].shift(1)


monthly_growth["orders_mom_pct"] = (
    (monthly_growth["orders"] - monthly_growth["previous_orders"]) /
    monthly_growth["previous_orders"]
) * 100 

print(monthly_growth)

       month  orders  previous_orders  orders_mom_pct
0 2018-04-01    6939              NaN             NaN
1 2018-05-01    6873           6939.0       -0.951146
2 2018-06-01    6167           6873.0      -10.272079
3 2018-07-01    6292           6167.0        2.026917
4 2018-08-01    6512           6292.0        3.496503


Monthly delivered revenue and cumulative revenue

In [ ]:
mask = (
    (orders["order_status"] == "delivered") & 
    (orders["order_purchase_timestamp"] >= '2018-04-01') & 
    (orders["order_purchase_timestamp"] < '2018-09-01')
)
filtered_orders = orders.loc[mask]

merged_orders = filtered_orders.merge(
    order_items,
    how="left",
    on="order_id"
)

monthly_revenue = (
    merged_orders.set_index("order_purchase_timestamp")
    .resample("MS")["price"]
    .sum()
    .reset_index()
)
monthly_revenue.columns = ["month", "revenue"]

monthly_revenue["cumulative_revenue"] = monthly_revenue["revenue"].cumsum()

print(monthly_revenue)

       month    revenue  cumulative_revenue
0 2018-04-01  973534.09           973534.09
1 2018-05-01  977544.69          1951078.78
2 2018-06-01  856077.86          2807156.64
3 2018-07-01  867953.46          3675110.10
4 2018-08-01  838576.64          4513686.74
